# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'external product page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 9 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'external company',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'endpoints product page',
   'url': 'https://endpoints.huggingface.co'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'learn resources page', 'url': 'https://huggingface.co/learn'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result



In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
5 days ago
•
382k
•
3.4k
openai/privacy-filter
Updated
10 days ago
•
99.4k
•
1.2k
XiaomiMiMo/MiMo-V2.5-Pro
Updated
4 days ago
•
9.91k
•
366
Qwen/Qwen3.6-27B
Updated
9 days ago
•
1.07M
•
1.07k
deepseek-ai/DeepSeek-V4-Flash
Updated
5 days ago
•
346k
•
918
Browse 2M+ models
Spaces
Running
on
Zero
MCP
2.51k
Wan2.2 14B Preview
🐌
2.51k
generate a video from an image with a text prompt
Running
on
CPU Upgrade
274
ML Intern
🤖
274
Ask ML questions and get instant helpful answers
Running
on
Zero
Agents
Featured
1.54k
TRELLIS.2
🏢
1.54k
High-fidelity 3D Generation from images
Running
on
Zero
MCP
884
Wan2.

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\n5 days ago\n•\n382k\n•\n3.4k\nopenai/privacy-filter\nUpdated\n10 days ago\n•\n99.4k\n•\n1.2k\nXiaomiMiMo/MiMo-V2.5-Pro\nUpdated\n4 days ago\n•\n9.91k\n•\n367\nQwen/Qwen3.6-27B\nUpdated\n9 days ago\n•\n1.07M\n•\n1.07k\ndeepseek-ai/DeepSeek-V4-Flash\nUpdated\n5 days ago\n•\n346k\n•\n918\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nMCP\n2.51k\nWan2.2 

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Hugging Face Brochure

## About Hugging Face
Hugging Face is the AI community building the future of machine learning. It is a vibrant collaboration platform where the machine learning community comes together to create, discover, and share models, datasets, and applications. Hugging Face empowers engineers, researchers, and end-users to innovate faster with open-source tools and a supportive ecosystem.

## Our Platform
- **Models:** Access and contribute to over 2 million machine learning models spanning multiple modalities — including text, image, video, audio, and even 3D data.
- **Datasets:** Browse and collaborate on more than 500,000 datasets to fuel your machine learning projects.
- **Spaces:** Host and explore AI applications running on cutting-edge infrastructure to test and demonstrate models.
- **Buckets:** Store and manage your data efficiently on the cloud.

Our platform encourages collaboration by enabling unlimited public hosting of models, datasets, and applications, allowing the community to move faster and build together.

## Community and Collaboration
Hugging Face fosters an open and ethical AI future, driven by a fast-growing community. The Hub serves as the central place for machine learning practitioners worldwide to:
- Share their work and build an ML portfolio
- Experiment with cutting-edge open-source ML technology
- Learn and collaborate across teams and disciplines

Trending projects and models demonstrate active participation, such as popular models in privacy filtering and advanced image and video generation.

## Enterprise Solutions
For organizations, Hugging Face provides paid Compute and Enterprise solutions designed to accelerate machine learning workflows with scalable infrastructure and support.

## Company Culture
Hugging Face is founded on openness, community empowerment, and shared progress in AI. The platform embodies a collaborative spirit where knowledge and resources are freely exchanged to help machine learning evolve ethically and inclusively.

## Careers
Join the Hugging Face team and contribute to the future of AI. By working here, you become part of a passionate community shaping an open AI ecosystem that benefits researchers, professionals, and users alike. Keep an eye on the Hugging Face website for open positions and internship programs such as ML Intern roles that offer hands-on experience with real-world AI challenges.

## Brand Identity
- **Colors:** Bright and energetic yellow (#FFD21E, #FF9D00) coupled with sleek gray tones (#6B7280) representing innovation and approachability.
- **Logo:** The distinctive Hugging Face logo symbolizes connection, friendliness, and collaboration in the AI community.

---

### Connect & Explore
Discover more or get involved:
- Website: [huggingface.co](https://huggingface.co)  
- Browse models and datasets  
- Sign up to start sharing and building your ML portfolio  
- Explore AI apps and demos in Spaces  
- Engage with a thriving global community committed to making AI accessible and ethical

---

**Hugging Face** — Empowering the next generation of machine learning through open collaboration and innovation.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [20]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [22]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Hugging Face: The AI Community Building the Future

---

## About Hugging Face

Hugging Face is the leading collaboration platform for the machine learning (ML) community worldwide. It serves as a vibrant hub where ML engineers, scientists, and enthusiasts come together to create, share, explore, and advance open-source AI models, datasets, and applications. Hugging Face empowers developers and organizations to accelerate their AI innovation and build the future of ethical machine learning.

---

## Our Platform

- **Extensive Model Library:** Access and browse over **2 million machine learning models** spanning text, image, video, audio, and 3D modalities. From natural language processing to cutting-edge generative models, the library is continuously updated by a fast-growing community.
  
- **Datasets:** Discover and contribute to over **500,000 datasets**, providing rich resources for research and development in AI.

- **Spaces:** Host and share interactive ML applications powered by the community. Spaces enable users to run AI demos and collaborate through engaging interfaces.

- **Open Source:** Benefit from the robust HF Open Source stack designed to help you build, integrate, and deploy ML solutions with speed and reliability.

- **Enterprise Solutions:** Hugging Face offers premium compute and enterprise-grade tools to scale AI workloads securely and efficiently.

---

## Community & Collaboration

- **Inclusive & Ethical AI:** Hugging Face is committed to fostering an **open, ethical, and collaborative AI ecosystem** where knowledge sharing and responsible innovation thrive.

- **Creators & Contributors:** The platform supports creators to build their portfolios and share their work globally, accelerating careers and advancing collective AI expertise.

- **Trending Projects:** Weekly curated highlights of popular models and applications showcasing the innovation of the community.

---

## Who Uses Hugging Face?

- **ML Engineers and Researchers:** Leverage state-of-the-art models and datasets for experimentation and production.

- **Enterprises:** Integrate best-in-class AI solutions tailored to their business needs with enterprise-grade support.

- **Developers & Innovators:** Harness the community’s work to build new AI-powered products and services.

- **Educators & Learners:** Learn and teach machine learning with access to a wealth of open-source resources and tools.

---

## Careers & Opportunities

Hugging Face nurtures a culture of innovation, openness, and ethical AI development. The company continually seeks passionate individuals who want to contribute to the future of machine learning and AI.

- **Join a Collaborative Community:** Work alongside top-tier AI researchers and engineers.
  
- **Build Impactful AI:** Influence projects that have a global reach in the AI community.

- **Grow Your Career:** Engage with cutting-edge open-source initiatives and expand your professional portfolio.

Explore career opportunities on the Hugging Face website under the "Jobs" or "Careers" sections and become part of the AI revolution.

---

## Brand & Identity

- **Logo & Colors:** Designed to reflect approachability and innovation with a vibrant palette of yellow (#FFD21E, #FF9D00) and neutral grey (#6B7280).

- **Mission:** To “Build the Future” by making AI technology collaborative, accessible, and safe for everyone.

---

## Get Involved

- **Sign Up:** Create your free account to start exploring AI applications and build your portfolio.
  
- **Explore & Contribute:** Share your models, upload datasets, launch Spaces, or collaborate on new projects.

- **Discover AI Apps:** Try out interactive AI-powered apps built by the community, pushing the boundaries of what machine learning can achieve.

---

Join **Hugging Face** today and be part of the thriving AI community shaping tomorrow’s technology!

Website: [huggingface.co](https://huggingface.co)

---

*Hugging Face – The AI community building the future.*

In [23]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. Serving as a leading collaboration platform, Hugging Face empowers machine learning engineers, scientists, and enthusiasts worldwide to share, discover, explore, and experiment with models, datasets, and applications. As a central hub for open-source ML, the platform fuels innovation and facilitates an open and ethical AI future.

---

## What We Offer

- **Models**  
  Access and browse over 2 million machine learning models spanning a wide range of modalities, including text, image, video, audio, and 3D.

- **Datasets**  
  Explore more than 500,000 datasets curated from global AI projects, enabling training and experimentation for countless applications.

- **Spaces & Applications**  
  Host and interact with over 1 million ML-powered applications and visuals — from 3D generation and video creation to instant ML Q&A assistants.

- **Compute & Enterprise Solutions**  
  Accelerate your machine learning projects with paid compute resources and enterprise-grade solutions tailored to scale your AI deployments efficiently.

- **Open Source Stack**  
  Build faster with Hugging Face’s open-source tools, libraries, and frameworks that support the entire ML workflow.

---

## Our Community & Customers

Hugging Face brings together a vibrant and fast-growing global community consisting of:

- Machine Learning Researchers and Practitioners  
- AI Engineers and Developers  
- Data Scientists  
- Enterprises leveraging AI solutions  
- AI Enthusiasts and Students  

The platform is the go-to space for collaborative innovation, where community members host, share, and advance ML research and applications openly and ethically.

---

## Company Culture

- **Collaborative and Open**  
  Hugging Face thrives on transparency and community-driven development, fostering a culture of sharing and mutual learning.

- **Inclusive and Ethical**  
  Committed to building responsible AI, the company promotes ethical standards and inclusivity in AI research and deployment.

- **Innovative and Impact-Driven**  
  With continuous innovation at its core, Hugging Face aims to enable faster progress in machine learning, empowering users to build the future of AI.

---

## Careers at Hugging Face

Join a passionate team working at the intersection of AI and community collaboration. If you want to be part of a company that:

- Pioneers open-source AI development  
- Collaborates globally with leading researchers and enterprises  
- Supports career growth in an inclusive environment  

Explore open roles for engineers, researchers, data scientists, and more. Make an impact on how the world builds and uses artificial intelligence.

---

## Connect with Us

- Visit: [huggingface.co](https://huggingface.co)  
- Sign up to create your ML portfolio and start collaborating today  
- Follow the latest trends with numerous updated models and datasets weekly  

---

Hugging Face — **The Home of Machine Learning Collaboration**  
Build, share, and accelerate your AI journey with the world’s leading machine learning community.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>